## isochrone fitting example

In [ ]:
import numpyro
num_chains = 4 # number of parallel MCMC chains
numpyro.set_host_device_count(num_chains)  # before JAX initializes devices

import jax
jax.config.update("jax_enable_x64", True)

from jaxstar.mistfit import (MistModel, MistFit, load_result,
                             plot_diagnostics, summarize, get_run_metadata)

In [ ]:
model = MistModel.from_isochrones()

In [ ]:
observations = {'teff': (5719, 110.), 'feh': (-0.12, 0.1), 'kmag': (8.843, 0.014), 'parallax': (7.27355841, 3.008e-2)}
# other magnitudes are also available
# observations = {'jmag': (6.258, 0.027), 'hmag': (5.576, 0.034), 'kmag': (5.364, 0.021), 'parallax': (2.1748371606513834, 0.03310597)}

In [ ]:
mf = MistFit(model=model, observations=observations)

### No-U-Turn Sampler (NUTS)

In [ ]:
mcmc = mf.run_nuts(
    jax.random.PRNGKey(0),
    num_chains=num_chains,
    num_warmup=5000,
    num_samples=5000,
    target_accept_prob=0.98,
    model_kwargs={
        "linear_age": True,
        "flat_age_marginal": False,
    },
    output_keys=("kmag_abs"),  
    )

In [ ]:
mcmc.print_summary()

In [ ]:
# Choose a new output directory for each run. Use `overwrite=True` explicitly to replace 
# an existing saved result or figure. The cells below preserve chain dimensions and do not rerun sampling.
idata = mf.to_inference_data(mcmc)
result_dir = mf.save_result(idata, "results/isochrone_nuts", overwrite=True)

In [ ]:
outkeys = list(mf.observations.keys) + ['feh_init', 'mass', 'radius', 'age', 'eep']
figures = mf.plot_diagnostics(idata, output_dir=result_dir, corner_kwargs={"keys": outkeys}, overwrite=True)

In [ ]:
# This also works in a fresh session without a MistModel or grid.
loaded = load_result("results/isochrone_nuts")
figures = plot_diagnostics(loaded, keys=("mass", "radius", "age"))
summarize(loaded, keys=("mass", "radius", "age"))
get_run_metadata(loaded)

## MultiNest (experimental)

Nested sampling is typically slower than NUTS but can explore multimodal posteriors more effectively. 
We recommend using at least 400 live points (n_live_points >= 400) and increasing this number to check the stability of the results.

In [ ]:
mn_dir = "results/isochrone_multinest"
mn = mf.run_multinest(outputfiles_basename=mn_dir + "/run1_", n_live_points=100)

In [ ]:
mn_idata = mf.to_inference_data(mn)
mf.save_result(mn_idata, mn_dir, overwrite=True)
mn_figures = mf.plot_diagnostics(mn_idata, output_dir=mn_dir,
                                keys=("mass", "radius", "age", "feh_init"),
                                overwrite=True)
get_run_metadata(mn_idata)["evidence"]